# Course 1, Module 2: Risk Buffers
## Where Should DDC Deploy Fogging Teams?

**Scenario:** DDC has confirmed 30 dengue cases across three provinces. Mosquito fogging teams need deployment zones. Budget is limited.

**Key Questions:**
1. Where are the risk rings around each case?
2. Which schools and hospitals fall inside those rings?
3. What is the total area that needs fogging?
4. How do we prioritize by severity tier?

In [1]:
from ddc_helpers import run_query, show_on_map, show_buffers

---

## Part 1: Risk Buffer Concept

### How Buffers Work

DDC uses **3 risk tiers** for vector control:
- **RED** (0-500m): Immediate fogging, larval survey
- **ORANGE** (500m-1km): Fogging within 48 hours
- **YELLOW** (1-2km): Enhanced surveillance, community awareness

### Important: GEOMETRY vs. GEOGRAPHY

`ST_Buffer()` only works with **GEOMETRY** (flat coordinates), not **GEOGRAPHY** (lat/lon on Earth's surface).

Since our data is stored as `GEOGRAPHY`, we convert using:
```sql
ST_GeomFromText(ST_AsText(geog))
```

Buffer distances are in **degrees**. At Thai latitudes (~14°N):
- **1 degree ≈ 111,000 meters**

For precise distance checks (e.g., "schools within 500m"), we still use `ST_Distance()` on GEOGRAPHY.

In [2]:
# Let's see what a single 500m buffer looks like
run_query("""
SELECT case_id, severity,
    ST_AsText(ST_Buffer(ST_GeomFromText(ST_AsText(geog)), 500.0 / 111000)) AS buffer_500m_wkt
FROM ddc_training.dengue_cases
WHERE case_id = 1
""")

,case_id,severity,buffer_500m_wkt
0,1,DF,"POLYGON ((100.557504505 13.723, 100.557417952 ..."


,case_id,severity,buffer_500m_wkt
0,1,DF,"POLYGON ((100.557504505 13.723, 100.557417952 ..."


**Explanation:** That WKT polygon is a circle with ~500m radius around case #1. In a GIS viewer, this would appear as a red ring around the patient's location.

---

## Part 2: Schools in Danger Zone

DDC sends inspectors to schools inside risk buffers to check for *Aedes aegypti* breeding containers (water jars, tires, flower pots).

We use `ST_Distance()` on GEOGRAPHY for accurate meter-based distances.

In [3]:
# Which schools are near dengue cases?
run_query("""
SELECT
    s.name AS school_name, s.district,
    d.case_id, d.severity, d.infection_date,
    ROUND(ST_Distance(s.geog, d.geog), 0) AS distance_meters,
    CASE
        WHEN ST_Distance(s.geog, d.geog) <= 500 THEN 'RED (0-500m)'
        WHEN ST_Distance(s.geog, d.geog) <= 1000 THEN 'ORANGE (500m-1km)'
        WHEN ST_Distance(s.geog, d.geog) <= 2000 THEN 'YELLOW (1-2km)'
    END AS risk_tier
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geog, d.geog) <= 2000
ORDER BY s.name, distance_meters
""")

,school_name,district,case_id,severity,infection_date,distance_meters,risk_tier
0,Benchama Rajalai School,Phra Nakhon,8,DF,2024-07-15,6085.0,NaN
1,Benchama Rajalai School,Phra Nakhon,3,DHF,2024-06-22,6234.0,NaN
2,Benchama Rajalai School,Phra Nakhon,12,DF,2024-08-15,6496.0,NaN
3,Benchama Rajalai School,Phra Nakhon,6,DF,2024-07-08,6498.0,NaN
4,Benchama Rajalai School,Phra Nakhon,1,DF,2024-06-15,6533.0,NaN
...,...,...,...,...,...,...,...
115,Wat Khlong Toei School,Khlong Toei,4,DF,2024-07-01,469.0,RED (0-500m)
116,Wat Khlong Toei School,Khlong Toei,3,DHF,2024-06-22,469.0,RED (0-500m)
117,Wat Khlong Toei School,Khlong Toei,7,DHF,2024-07-12,474.0,RED (0-500m)
118,Wat Khlong Toei School,Khlong Toei,8,DF,2024-07-15,623.0,ORANGE (500m-1km)


,school_name,district,case_id,severity,infection_date,distance_meters,risk_tier
0,Benchama Rajalai School,Phra Nakhon,8,DF,2024-07-15,6085.0,NaN
1,Benchama Rajalai School,Phra Nakhon,3,DHF,2024-06-22,6234.0,NaN
2,Benchama Rajalai School,Phra Nakhon,12,DF,2024-08-15,6496.0,NaN
3,Benchama Rajalai School,Phra Nakhon,6,DF,2024-07-08,6498.0,NaN
4,Benchama Rajalai School,Phra Nakhon,1,DF,2024-06-15,6533.0,NaN
...,...,...,...,...,...,...,...
115,Wat Khlong Toei School,Khlong Toei,4,DF,2024-07-01,469.0,RED (0-500m)
116,Wat Khlong Toei School,Khlong Toei,3,DHF,2024-06-22,469.0,RED (0-500m)
117,Wat Khlong Toei School,Khlong Toei,7,DHF,2024-07-12,474.0,RED (0-500m)
118,Wat Khlong Toei School,Khlong Toei,8,DF,2024-07-15,623.0,ORANGE (500m-1km)


**Insight:** Schools that appear multiple times are near **multiple cases**. These are highest priority for DDC inspection teams.

In [4]:
# Aggregate: How many cases threaten each school?
run_query("""
SELECT
    s.name AS school_name, s.district,
    COUNT(*) AS threatening_cases,
    SUM(CASE WHEN d.severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_nearby,
    MIN(ROUND(ST_Distance(s.geog, d.geog), 0)) AS nearest_case_meters
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geog, d.geog) <= 2000
GROUP BY s.name, s.district
ORDER BY threatening_cases DESC
""")

,school_name,district,threatening_cases,severe_nearby,nearest_case_meters
0,Benchama Rajalai School,Phra Nakhon,12,4,6085.0
1,Khlong Toei Wittaya School,Khlong Toei,12,4,275.0
2,Suankularb Wittayalai School,Phra Nakhon,12,4,6167.0
3,Din Daeng Wittaya School,Din Daeng,12,4,4661.0
4,Sukhumvit Pattana School,Watthana,12,4,850.0
5,Ratchadaphisek Wittayalai School,Din Daeng,12,4,4072.0
6,Wat Khlong Toei School,Khlong Toei,12,4,124.0
7,Satri Witthaya School,Dusit,12,4,6491.0
8,Triam Udom Suksa School,Pathum Wan,12,4,2389.0
9,Huai Khwang School,Huai Khwang,12,4,4972.0


,school_name,district,threatening_cases,severe_nearby,nearest_case_meters
0,Benchama Rajalai School,Phra Nakhon,12,4,6085.0
1,Khlong Toei Wittaya School,Khlong Toei,12,4,275.0
2,Suankularb Wittayalai School,Phra Nakhon,12,4,6167.0
3,Din Daeng Wittaya School,Din Daeng,12,4,4661.0
4,Sukhumvit Pattana School,Watthana,12,4,850.0
5,Ratchadaphisek Wittayalai School,Din Daeng,12,4,4072.0
6,Wat Khlong Toei School,Khlong Toei,12,4,124.0
7,Satri Witthaya School,Dusit,12,4,6491.0
8,Triam Udom Suksa School,Pathum Wan,12,4,2389.0
9,Huai Khwang School,Huai Khwang,12,4,4972.0


---

## Part 3: Visualize Risk Buffers on Map

In [5]:
# Show all risk buffers on an interactive map
show_buffers("""
SELECT
    province_name, risk_tier, radius_meters, case_count,
    area_sq_km, ST_AsText(zone_geom) AS wkt
FROM ddc_training.dengue_risk_zones
ORDER BY radius_meters
""")

---

## Part 4: Area Calculation

### Converting Square Degrees to Square Kilometers

`ST_Area()` on GEOMETRY returns results in **square degrees**.

Conversion formula at Thai latitude (~14°N):
- 1 degree latitude ≈ 111 km
- 1 degree longitude ≈ 108 km (cos(14°) × 111 km)
- **1 square degree ≈ 111 × 108 ≈ 11,988 square km**

**Multiply square degrees by 11,988 to get square kilometers.**

In [6]:
# Total area needing treatment by province and risk tier
run_query("""
SELECT
    province_name, risk_tier, case_count,
    area_sq_km
FROM ddc_training.dengue_risk_zones
ORDER BY province_name, radius_meters
""")

,province_name,risk_tier,case_count,area_sq_km
0,Bangkok,RED,20,3.963
1,Bangkok,ORANGE,20,10.702
2,Bangkok,YELLOW,20,33.245
3,Chiang Mai,RED,6,1.745
4,Chiang Mai,ORANGE,6,4.981
5,Chiang Mai,YELLOW,6,15.983
6,Chiang Rai,RED,4,1.362
7,Chiang Rai,ORANGE,4,4.252
8,Chiang Rai,YELLOW,4,14.585


,province_name,risk_tier,case_count,area_sq_km
0,Bangkok,RED,20,3.963
1,Bangkok,ORANGE,20,10.702
2,Bangkok,YELLOW,20,33.245
3,Chiang Mai,RED,6,1.745
4,Chiang Mai,ORANGE,6,4.981
5,Chiang Mai,YELLOW,6,15.983
6,Chiang Rai,RED,4,1.362
7,Chiang Rai,ORANGE,4,4.252
8,Chiang Rai,YELLOW,4,14.585


---

## Part 5: Overlap Savings

### Why Merge Buffers?

**Problem:** When cases are clustered, their 500m buffers overlap. Fogging teams don't need to spray the same area twice.

**Solution:** `ST_Union()` merges overlapping polygons into a single shape.

**Benefit:** Compare the "naive sum" (all circles, with overlap) vs. the "merged area" (overlap removed). The difference is **overlap savings**—resources saved by smart fogging deployment.

In [7]:
# Calculate overlap savings per province
run_query("""
SELECT
    pb.province_name,
    ROUND(SUM(ST_Area(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000)))*11988, 3) AS naive_sum_sq_km,
    ROUND(ST_Area(ST_Union(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000)))*11988, 3) AS merged_sq_km,
    ROUND((1 - ST_Area(ST_Union(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000)))
         / NULLIFZERO(SUM(ST_Area(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000))))) * 100, 1) AS overlap_savings_pct
FROM ddc_training.dengue_cases d
JOIN ddc_training.province_boundaries pb
    ON ST_Intersects(ST_GeomFromText(ST_AsText(pb.boundary)), ST_GeomFromText(ST_AsText(d.geog)))
GROUP BY pb.province_name
ORDER BY overlap_savings_pct DESC
""")

,province_name,naive_sum_sq_km,merged_sq_km,overlap_savings_pct
0,Bangkok,15.185,3.963,73.9
1,Chiang Mai,4.556,1.745,61.7
2,Chiang Rai,3.037,1.362,55.2


,province_name,naive_sum_sq_km,merged_sq_km,overlap_savings_pct
0,Bangkok,15.185,3.963,73.9
1,Chiang Mai,4.556,1.745,61.7
2,Chiang Rai,3.037,1.362,55.2


---

## Key Functions Summary

| Function | What it does |
|---|---|
| `ST_Buffer(geom, degrees)` | Creates a circle polygon at given radius |
| `ST_Intersects(a, b)` | TRUE if two spatial objects overlap |
| `ST_Union(geom)` | Merges multiple polygons, removes overlap |
| `ST_Area(geom)` | Area in square degrees (GEOMETRY) |
| `ST_Distance(geog, geog)` | Proximity check in meters (GEOGRAPHY) |
| `ST_GeomFromText(ST_AsText(geog))` | Convert GEOGRAPHY to GEOMETRY for buffering |
| `ST_AsText(geom)` | Convert geometry to WKT text format |

---

## Exercise: Fogging Priority Ranking

### Your Task

Create a **priority ranking for fogging deployment** by calculating a "risk score" per school:

**Scoring Rules:**
- **3 points** for each RED-tier case nearby (< 500m)
- **2 points** for each ORANGE-tier case (500m - 1km)
- **1 point** for each YELLOW-tier case (1km - 2km)
- **Double the points** if case severity is DHF or DSS (severe dengue)

**Output:** Rank schools by risk score (highest first).

**Questions to Answer:**
1. Which 3 schools get fogging first?
2. What is the average risk score?
3. How many schools are in the RED zone?

**Hints:**
- Use nested CASE statements to assign points based on distance tiers
- Use `SUM()` with `CASE` to double severe cases (DHF/DSS)
- Join `schools`, `dengue_cases`, and use `ST_Distance()` for distance calculations
- GROUP BY school, then ORDER BY risk score DESC

In [ ]:
# Write your SQL query below
# Use run_query() to execute it

run_query("""
-- Your SQL here

""")

---

### Model Answer

<details>
<summary><b>Click to reveal the answer</b></summary>

```sql
SELECT
    s.name AS school_name,
    s.district,
    COUNT(*) AS case_count,
    SUM(
        CASE
            WHEN ST_Distance(s.geog, d.geog) <= 500 THEN 3
            WHEN ST_Distance(s.geog, d.geog) <= 1000 THEN 2
            WHEN ST_Distance(s.geog, d.geog) <= 2000 THEN 1
        END *
        CASE
            WHEN d.severity IN ('DHF', 'DSS') THEN 2
            ELSE 1
        END
    ) AS risk_score
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geog, d.geog) <= 2000
GROUP BY s.name, s.district
ORDER BY risk_score DESC, case_count DESC
```

**Expected Results:**
- Top schools by risk score represent highest-priority fogging zones
- Schools with severe cases (DHF/DSS) nearby score higher
- Multiple threats in RED zone amplify priority

</details>